In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [3]:
df = pd.read_csv('homework_7.1.csv')
df.drop(columns=['Unnamed: 0'], inplace=True)
df

,X,W,Z,Y
0,1.137055,1.221768,0.327829,1.944532
1,-0.112905,0.465835,0.599650,0.655514
2,2.077755,1.795414,-0.063393,5.934411
3,0.456373,-0.512159,1.177413,-0.188064
4,-1.012402,0.080002,-0.275697,-0.533775
...,...,...,...,...
9995,2.569934,1.233620,0.930467,6.188783
9996,0.190150,1.022164,-0.015151,0.697187
9997,-1.184465,-1.475929,-0.287056,-1.575303
9998,-0.121286,-0.914357,1.706237,-1.809819


Q1. Suppose that a process can be modeled via linear regression: 

```
W = np.random.normal(0, 1, (1000,))
X = W + np.random.normal(0, 1, (1000,)) 
Z = np.random.normal(0, 1, (1000,)) 
Y = X + Z + W + np.random.normal(0, 1, (1000,))
```

Which is closest to the correlation of *X* with the error term in the equation for *Y*? 

Option A: -0.50

Option B: 0

Option C: 1.0

Option D: 0.75

Option E: 0.5


In [4]:
W = np.random.normal(0, 1, (1000,))
X = W + np.random.normal(0, 1, (1000,)) 
Z = np.random.normal(0, 1, (1000,)) 
Y = X + Z + W + np.random.normal(0, 1, (1000,))

In [5]:
structural_error = Y - X
np.corrcoef(X, structural_error)

array([[1.        , 0.41080479],
       [0.41080479, 1.        ]])

A1. Option E (0.5)

Q2: If *Y* is written as depending on *X* and *Z* only, *W* is part of the error term. Which, then, is closest to the expected correlation of *X* with the error term in the equation for *Y*?

Option A: 0.75

Option B: 0.50

Option C: 1.0

Option D: -0.50

Option E: 0

In [6]:
structural_error = Y - X - Z

# Calculate the correlation between X and the structural error
corr_matrix = np.corrcoef(X, structural_error)
corr_matrix

array([[1.       , 0.4988745],
       [0.4988745, 1.       ]])

A2. Option B. (0.5)

Q3. In the data frame for homework_7.1.csv, control for W by regressing *Y* on X
 and ﻿Z﻿ at the following constant values of ﻿W﻿: -1, 0, and 1. (You cannot literally use a constant value of ﻿W﻿, of course, or you will have only one data point! How will you manage this?) The question is: Is the coefficient of ﻿X﻿

Option A. decreasing

Option B. staying about the same (say, within 0.2 or so) as﻿W﻿ increases? 

Option C. increasing

In [7]:
# 2. Define a narrow window to simulate holding W constant
tolerance = 0.05
w_targets = [-1, 0, 1]

print("--- Regression Results within W Slices ---")

for w_val in w_targets:
    # Create a boolean mask for the current slice (e.g., -1.05 <= W <= -0.95)
    mask = (W >= w_val - tolerance) & (W <= w_val + tolerance)
    
    # Isolate variables inside the slice
    X_slice = X[mask]
    Z_slice = Z[mask]
    Y_slice = Y[mask]
    
    # Set up and fit the OLS regression: Y = b0 + b1*X + b2*Z
    features = np.column_stack((X_slice, Z_slice))
    features = sm.add_constant(features)
    model = sm.OLS(Y_slice, features).fit()
    
    # params[0] is intercept, params[1] is X coefficient, params[2] is Z coefficient
    x_coef = model.params[1] 
    sample_count = np.sum(mask)
    
    print(f"Slice W ≈ {w_val:2d} (n={sample_count:,}) | X Coefficient: {x_coef:.4f}")

--- Regression Results within W Slices ---
Slice W ≈ -1 (n=25) | X Coefficient: 0.8873
Slice W ≈  0 (n=40) | X Coefficient: 0.8076
Slice W ≈  1 (n=25) | X Coefficient: 0.7159


A3. Option B.

Q4. 

```
def make_error(corr_const, num): 
          err = list() 
          prev = np.random.normal(0, 1) 
          for n in range(num): 
                   prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 
                   err.append(prev) 
          return np.array(err) 
```

Create a linear regression model that uses this function as the error for both (a) the treatment, ﻿X﻿, and (b) the outcome, ﻿Y﻿. (You can use random normal error for any other covariates, if you have them.) 

As corr_const increases from 0.2 to 0.5 to 0.8, find (i) the standard deviation of the estimate of the ﻿X﻿ coefficient over many trials, and (ii) the mean of the standard error estimate of the ﻿X﻿ coefficient over many trials. 

When corr_const increases, then: 

Hint: don't forget to include an intercept in your regression

Option A: (i) and (ii) differ, but their ratio remains about the same

Option B: The ratio (i) / (ii) decreases

Option C: The ratio (i) / (ii) increases

Option D: (i) and (ii) remain about the same

In [8]:
def make_error(corr_const, num): 
          err = list() 
          prev = np.random.normal(0, 1) 
          for n in range(num): 
                   prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 
                   err.append(prev) 
          return np.array(err) 

In [9]:
# Run 1,000 trials to calculate (i) actual SD and (ii) mean OLS Standard Error
for c in [0.2, 0.5, 0.8]:
    estimates, std_errors = [], []
    for _ in range(1000):
        X = make_error(c, 1000)
        Y = X + make_error(c, 1000)  # DGP: Y = 1*X + error
        
        model = sm.OLS(Y, sm.add_constant(X)).fit()
        estimates.append(model.params[1])
        std_errors.append(model.bse[1])
        
    actual_sd = np.std(estimates)
    mean_se = np.mean(std_errors)
    print(f"corr_const = {c} | (i) Actual SD: {actual_sd:.4f} | (ii) Mean SE: {mean_se:.4f} | Ratio: {actual_sd/mean_se:.2f}")

corr_const = 0.2 | (i) Actual SD: 0.0340 | (ii) Mean SE: 0.0317 | Ratio: 1.07
corr_const = 0.5 | (i) Actual SD: 0.0401 | (ii) Mean SE: 0.0316 | Ratio: 1.27
corr_const = 0.8 | (i) Actual SD: 0.0678 | (ii) Mean SE: 0.0316 | Ratio: 2.14


A4. Option C

Reflection 7 Problem 1

Create a linear regression model involving a confounder that is left out of the model.  Show whether the true correlation between $$X$$ and $$Y$$ is overestimated, underestimated, or neither.  Explain in words why this is the case for the given coefficients you have chosen.


In [48]:
num = 10_000

# Simulate the lightning storm which has the first effects caused by it. We could model it as a 
# float (0 == no storm and from there it is a severity of a storm). 
# This seemed complicated so I make it binary instead (Yes or No storm) with equal prob
lightning_storm = np.random.binomial(1, 0.5, num)

# Lightning storms frighten bears, decreasing their population
# Create baseline population noise
bears = np.random.normal(1_000, 50, num) # Normal distribution centered around 1,000 with std 50
# Add effect of lighting storm = 1
bears = np.where(lightning_storm == 1, bears * 0.9, bears) # When lightning storm happens it decreases bears population by 10%

# Lightning storm frighten deer, decreasing their population
deer = np.random.normal(1_000, 50, num) # Normal distribution centered around 1,000 with std 50
deer = np.where(lightning_storm == 1, deer * 0.9, deer) # When lightning storm happens it decreases deer population by 10%

# Lightning storm cause flowers to grow, increasing their population.
flowers = np.random.normal(1_000, 50, num) # Normal distribution centered around 1,000 with std 50
flowers = np.where(lightning_storm == 1, flowers + 100, flowers) # When lightning storm happens it decreases deer by 100 (static value)

# Bears eat deer, decreasing their population.
deer = deer - (bears * 0.1) # Decrease deer count by 10% of the bear population. (random number I decided on shows effect)

# Deer eat flowers, decreasing their population.
flowers = flowers - (deer * 0.1) # Decrease flowers count by 10% of the deer population. (random number I decided on)

df = pd.DataFrame(
    {
        'lightning_storm': lightning_storm,
        'bears': bears,
        'deer': deer,
        'flowers': flowers
    }
)
df

,lightning_storm,bears,deer,flowers
0,1,959.509092,870.291852,976.940195
1,1,912.070759,863.658119,969.903447
2,1,856.451453,736.343375,1068.911860
3,1,930.316314,764.468099,1055.546899
4,1,899.304342,889.240552,1004.854483
...,...,...,...,...
9995,0,1073.377690,913.083492,864.117105
9996,1,836.659363,833.248502,938.215854
9997,1,893.046893,805.752479,952.388857
9998,1,841.745409,836.609270,1020.085861


In [ ]:
# The confounder is lightning_storm, the X is bears, and the Y is deer

X_without_lightning= sm.add_constant(df['bears'])
model_without_lightning = sm.OLS(df['deer'], X_without_lightning).fit()

X_with_lightning = sm.add_constant(df[['bears', 'lightning_storm']])
model_with_lightning = sm.OLS(df['deer'], X_with_lightning).fit()

model_without_lightning.params['bears'], model_with_lightning.params['bears']

(np.float64(0.41157357014404083), np.float64(-0.09848556025687241))

We know that the actual coef is -0.10 (from the *0.1 calculation in sim code)

Without the confounder we estimate 0.41157357014404083, overestimating

With the confounder we estimate 0.09848556025687241 which is close to the true value